# 1. Research and dataset selection

**Goal:** find public customer-agent voice data, compare it, and choose with explicit criteria.
The full survey is in `reports/research_survey.xlsx`; this notebook reads it and re-runs the sampling.

**Key fact to keep in view:** the selected data is **role-played, not real customer calls**.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, Markdown, display

from calltrace.config import PROJECT_ROOT, load_config
from calltrace.data import AppTekDataset

pd.set_option("display.max_colwidth", 160, "display.max_rows", 80)
config = load_config()
dataset = AppTekDataset(config)
results = config.path("results")
evaluation_dir = results / "evaluation"


## Dataset survey
Every row was checked on its source page. `Unknown` means the source does not say.

In [2]:
survey_path = config.path("reports") / "research_survey.xlsx"
datasets = pd.read_excel(survey_path, sheet_name="Dataset comparison", header=3).dropna(subset=["Source"])
datasets = datasets[datasets["ID"].astype(str).str.startswith("D")]
datasets[["ID", "Source", "Number of calls", "Audio availability", "Transcript availability", "License / usage conditions", "Recommendation"]]

,ID,Source,Number of calls,Audio availability,Transcript availability,License / usage conditions,Recommendation
0,D1,"AppTek Call-Center Dialogues (AppTek.ai; Beck et al., 2026)","873 calls (1,746 channel files)","Yes. 16 kHz WAV, one speaker per file, plus a merged file per call","Yes. Fully manual, with timed segments in the diarization config","CC BY-SA 4.0. Card: intended for evaluation and analysis, not for training ASR or foundation models",SELECTED
1,D2,"SpokenWOZ (Si et al., NeurIPS 2023)","5,700 dialogues","Yes. Two-track, 8 kHz; test archive 2.18 GB, train/dev archive 10.34 GB",Partial. ASR-generated. Paper: agent text was corrected by annotators; user text keeps ASR noise on purpose,CC BY-NC 4.0 (project page),Runner-up. Best alternative if channel audio with larger scale matters more than clean references
2,D3,"CallCenterEN (AIxBlock; Dao et al., 2025)","91,706 conversations",No. Paper: audio withheld for biometric privacy reasons,Yes. Machine-generated (AssemblyAI) with word timestamps; PII redacted,"CC BY-NC 4.0 tag; card: research and model development only, no commercial use, resale, or redistribution",Rejected: fails the audio requirement
3,D4,AxonData contact-center samples (English and multilingual cards),"Open sample: 2 English calls; multilingual sample: 11 calls. Full data (advertised 1,000+ h) is paid",Sample only. Mono MP3 from 8 kHz telephone speech,Sample only (DOCX). Multilingual card also mentions concise summaries,CC BY-NC 4.0 tag on the sample; full data needs a commercial agreement,Rejected: fewer than 10 usable English calls openly available
4,D5,UniData call-center-audio preview,"Open preview: 5 FLAC calls. Full data (13,000+ h advertised) is paid",Preview only,"Yes, in metainfo.csv of the preview",CC BY-NC-ND 4.0 tag (no derivatives); full data needs purchase,"Rejected: too small, derivative restriction, and responsible-use concern"
5,D6,InfoBayAI Arabic Call Center Audio (dual channel),Open listing shows 6 WAV files in sample folders (3 samples by file count). Full size Unknown,Gated: contact details must be shared first,Unknown,CC BY 4.0 tag on the sample; card and terms not readable without accepting the gate,Rejected: gated and too small. Worth revisiting for Arabic work
6,D7,Nakulbh/call-center-dataset,"20 WAV files in agent/customer folders, implying 10 calls","Yes, split agent/customer files",None found,Not stated: no license and no README found,"Rejected: no permitted use, possible private recordings"
7,D8,AIxBlock Japanese human-to-machine call-center audio,Unknown,Yes,Not stated on the card,CC BY-NC 4.0,Rejected: not human agent-customer speech
8,D9,"Arabic call-domain ASR benchmark (Obaidah et al., 2024)",Not stated in the sections reviewed,16 kHz mono per paper. No public download located,"Manual (41 annotators, 13 reviewers) per paper",Not stated: no data license located,Rejected: not accessible. Used as related work only
9,D10,Switchboard-1 Release 2 and CALLHOME American English (LDC),"About 2,400 (Switchboard); 120 (CALLHOME)","Yes, for LDC licensees. 2-channel, 8 kHz",Yes,LDC user agreement; paid or membership access,Rejected: not relevant and restricted


## Eligibility, then comparison
Audio access and permitted use are gates. Only datasets that pass all gates get a weighted score.

In [3]:
selection = pd.read_excel(survey_path, sheet_name="Selection decision", header=None)
gate_start = selection.index[selection[0] == "Dataset"][0]
gates = selection.iloc[gate_start + 1 : gate_start + 11]
gates.columns = selection.iloc[gate_start]
gates.iloc[:, :6]

4,Dataset,Audio accessible (no purchase or gate),Permitted use is clear,At least 10 customer-agent calls,Responsible use,Result
5,D1,Pass,Pass,Pass,Pass,Eligible
6,D2,Pass,Pass,Pass,Pass,Eligible
7,D3,FAIL: no audio,Pass,Pass,Pass,Not eligible
8,D4,Pass (sample),Pass (sample),FAIL: 2 English calls,Unclear: consent not documented,Not eligible
9,D5,Pass (preview),FAIL: no derivatives,FAIL: 5 calls,"FAIL: named companies, user location",Not eligible
10,D6,FAIL: gated,Unclear: terms unread,FAIL: about 3 calls,Unknown,Not eligible
11,D7,Pass,FAIL: no license,Unclear: content unknown,FAIL: provenance unknown,Not eligible
12,D8,Pass,Pass,FAIL: human-to-machine,Pass,Not eligible
13,D9,FAIL: not located,FAIL: not located,Unknown,Pass per paper,Not eligible
14,D10,FAIL: licensed,Pass (under license),FAIL: not service calls,Pass,Not eligible


In [4]:
score_start = selection.index[selection[0] == "Criterion"][0]
scores = selection.iloc[score_start + 1 : score_start + 12, :4]
scores.columns = selection.iloc[score_start, :4]
scores

17,Criterion,Weight,D1 AppTek score,D2 SpokenWOZ score
18,Customer-agent relevance,20,5,3
19,Usable speaker separation,20,5,4
20,Reference annotations for evaluation,20,5,2
21,Clear permitted use,15,5,4
22,Reproducible access,10,5,3
23,Manageable download and compute,10,5,4
24,Closeness to real calls,5,3,2
25,Weight total (should be 100),100,NaN,NaN
26,Weighted score (out of 5),NaN,4.9,3.2
27,Sensitivity: equal weights,NaN,4.714286,3.142857


## Papers and how they shaped the work

In [5]:
papers = pd.read_excel(survey_path, sheet_name="Papers and articles", header=3)
papers[["ID", "Reference", "How it is used here"]]

,ID,Reference,How it is used here
0,P1,Beck et al. (2026). AppTek Call-Center Dialogues: A Multi-Accent Long-Form Benchmark for English ASR.,"Chose Parakeet v3 and Silero settings; adopted the dataset's WER protocol; the published average flagged our int8 run as abnormal (14.39% vs 9.2%), which le..."
1,P2,Si et al. (2023). SpokenWOZ: A Large-Scale Speech-Text Benchmark for Spoken Task-Oriented Dialogue Agents. NeurIPS.,Runner-up in selection. Its backchannel discussion motivated separate backchannel handling in turn building.
2,P3,Jia & SungChu (2020). A Deep Learning System for Sentiment Analysis of Service Calls.,"Motivated the ASR-versus-reference experiment, the requirement for reliable speaker separation, and caution about single-turn evidence."
3,P4,Gao et al. (2023). Enabling Large Language Models to Generate Text with Citations. EMNLP.,Basis for separating 'citation exists' (mechanical check) from 'citation supports the claim' (grounding audit).
4,P5,Feigenblat et al. (2021). TWEETSUMM: A Dialog Summarization Dataset for Customer Service. Findings of EMNLP.,Supports the issue / action / outcome structure required by the assessment.
5,P6,An empirical analysis of word error rate and keyword error rate. INTERSPEECH 2008 (IBM Research).,"Reason to inspect which words are wrong (names, numbers), and to test downstream effects directly instead of relying on WER."
6,P7,Ghosh et al. (2025). ASR-Synchronized Speaker-Role Diarization.,Supports keeping Speaker 1/2 separate from Customer/Agent and documenting how each is obtained.


## Sampling
Rule: one call per accent, using a seeded shuffle, taking the first call whose domain is unused.
It reads only metadata, so no call can be chosen because it is easy. Running it again gives the same sample.

In [6]:
manifest = dataset.sample()
print(f"Catalogue: {manifest['catalogue_calls']} calls; eligible: {manifest['eligible_calls']}; exclusions: {len(manifest['exclusions'])}")
sample = pd.DataFrame(manifest["selected"])
sample["agent_channel"] = sample["channels"].map(lambda c: next(k for k, v in c.items() if v["role"] == "agent"))
sample[["call_id", "source_id", "accent", "domain", "duration_s", "agent_channel"]]

Catalogue: 873 calls; eligible: 873; exclusions: 0


,call_id,source_id,accent,domain,duration_s,agent_channel
0,001,en_AU_Banking_1590914,en-AU,banking,633.177,1
1,002,en_CA_RealEstate_1592209,en-CA,realestate,690.862,1
2,003,en_CN_Health_1584519,en-CN,health,519.861,1
3,004,en_GB_Travel_1582389,en-GB,travel,624.467,1
4,005,en_GB_SCT_Aviation_1593663,en-GB_SCT,aviation,358.734,1
5,006,en_GB_WLS_Energy_1593355,en-GB_WLS,energy,582.360,2
6,007,en_IE_Food_1592985,en-IE,food,902.865,1
7,008,en_IN_Entertainment_002_20250918,en-IN,entertainment,586.133,1
8,009,en_MX_DeliveryService_1587298,en-MX,deliveryservice,781.155,1
9,010,en_SG_Telecom_1586744,en-SG,telecom,888.000,1


## Data-quality checks on the catalogue

In [7]:
catalogue = dataset.build_catalogue()
channel1_customer = sum(r.channels["1"]["role"] == "customer" for r in catalogue)
print(f"{len(catalogue)} calls. All have one agent and one customer: {all({c['role'] for c in r.channels.values()} == {'agent', 'customer'} for r in catalogue)}")
print(f"Channel 1 is the customer in {channel1_customer} calls, so channel order cannot be treated as role.")
print(f"Total recording time: {sum(r.duration_s for r in catalogue) / 3600:.1f} h (the card's 128.6 h is annotated speech).")

873 calls. All have one agent and one customer: True
Channel 1 is the customer in 66 calls, so channel order cannot be treated as role.
Total recording time: 151.1 h (the card's 128.6 h is annotated speech).


In [8]:
from calltrace.data import audio_quality

rows = []
for record in manifest["selected"]:
    # Uses files that are already downloaded; downloads only what is missing.
    paths = dataset.download_call(record)
    for channel in sorted(record["channels"]):
        row = audio_quality(paths[f"channel{channel}"])
        rows.append({"call_id": record["call_id"], "channel": channel, "role": record["channels"][channel]["role"],
                     **{k: row[k] for k in ("samplerate", "duration_s", "clipped_share", "active_share", "speech_rms")}})
quality = pd.DataFrame(rows)
quality.to_csv(config.path("manifests") / "audio_quality.csv", index=False)
print("Sample rates:", sorted(quality.samplerate.unique()))
print("Largest clipped share:", quality.clipped_share.max())
quality.head(6)

Sample rates: [np.int64(16000)]
Largest clipped share: 1.6e-05


,call_id,channel,role,samplerate,duration_s,clipped_share,active_share,speech_rms
0,001,1,agent,16000,633.177,0.000003,0.3786,0.05004
1,001,2,customer,16000,633.177,0.000000,0.2673,0.05507
2,002,1,agent,16000,690.862,0.000000,0.3651,0.06391
3,002,2,customer,16000,690.862,0.000000,0.1993,0.03051
4,003,1,agent,16000,519.861,0.000000,0.3759,0.08293
5,003,2,customer,16000,519.861,0.000016,0.2945,0.12268


## Decision
**AppTek Call-Center Dialogues** is the only candidate with audio open to all, split channels, manual references for
both speakers, and a license that permits sharing derived results. Its main weakness, role-played speech, limits every
conclusion: results describe these role-plays and do not support claims about real customers or business impact.